# Language Modeling & Word Embeddings — Hands-On Notebook
Companion to: [Language Modeling](../docs/01_foundations/language_modeling_basics.md), [Word Embeddings](../docs/01_foundations/word_embeddings.md)

**What you'll build:**
- A bigram language model from scratch
- Perplexity calculation by hand
- Word2Vec skip-gram training on a tiny corpus
- Explore pretrained embeddings with gensim
- BPE tokenization with HuggingFace


## Section 1: Setup

**Dependencies:** `torch`, `matplotlib`, `scikit-learn`, `gensim`, `transformers`. Install if needed, e.g. `pip install torch transformers gensim scikit-learn matplotlib`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict
import math
import torch.nn.functional as F
import random

try:
    from sklearn.decomposition import PCA
except ImportError:
    PCA = None

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


## Section 2: What is a Language Model?

A **language model** assigns probabilities to sequences of words. The key idea:

$P(w_1, w_2, \ldots, w_n) = P(w_1) \times P(w_2|w_1) \times P(w_3|w_1,w_2) \times \cdots$

This is just the **chain rule** of probability.


In [ ]:
# Tiny corpus: 6 short sentences
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "a cat and a dog",
    "the mat sat near the cat",
    "on the log the dog sat",
    "the cat sat and the dog sat",
]

def tokenize(s):
    return s.lower().replace(".", "").split()

all_tokens = []
for s in corpus:
    all_tokens.extend(tokenize(s))

unigram_counts = Counter(all_tokens)
total_words = sum(unigram_counts.values())
V = len(unigram_counts)

def P_unigram(w):
    w = w.lower()
    return unigram_counts[w] / total_words if w in unigram_counts else 0.0

print("Vocabulary size:", V)
print("Unigram counts (sample):", dict(list(unigram_counts.most_common(8))))
print("P(the) =", P_unigram("the"))
print("P(cat) =", P_unigram("cat"))


In [ ]:
# Bigram counts: P(w2 | w1) = count(w1, w2) / count(w1 as first word in bigram)
bigram_counts = Counter()
unigram_context = Counter()

for s in corpus:
    toks = tokenize(s)
    for i in range(len(toks) - 1):
        w1, w2 = toks[i], toks[i + 1]
        bigram_counts[(w1, w2)] += 1
        unigram_context[w1] += 1

def P_bigram(w2_given_w1, w1):
    denom = unigram_context[w1]
    if denom == 0:
        return 0.0
    return bigram_counts[(w1, w2_given_w1)] / denom

print('P("cat" | "the") =', P_bigram("cat", "the"))
print("  count(the, cat) =", bigram_counts[("the", "cat")], ", count(the,*) =", unigram_context["the"])


In [ ]:
# Heatmap of P(w2 | w1) over vocabulary (MLE)
words = sorted(unigram_counts.keys())
n = len(words)
prob_matrix = np.zeros((n, n))
for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        prob_matrix[i, j] = P_bigram(w2, w1)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(prob_matrix, cmap="Blues", aspect="auto", vmin=0, vmax=prob_matrix.max() or 1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(words, rotation=45, ha="right")
ax.set_yticklabels(words)
ax.set_xlabel("Next word $w_2$")
ax.set_ylabel("Current word $w_1$")
ax.set_title("Bigram conditional probabilities P(w2 | w1), MLE")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Probability")
plt.tight_layout()
plt.show()


## Section 3: Generating Text with N-grams

Sample the next word from $P(\cdot \mid \text{current word})$ and chain **20** words.


In [ ]:
def sample_next_word(w1, rng=random):
    candidates = [w2 for (a, w2) in bigram_counts if a == w1]
    if not candidates:
        return None
    weights = [bigram_counts[(w1, w2)] for w2 in candidates]
    return rng.choices(candidates, weights=weights, k=1)[0]

def generate_bigram(start="the", n_words=20, rng=random):
    out = [start]
    for _ in range(n_words - 1):
        w = sample_next_word(out[-1], rng=rng)
        if w is None:
            break
        out.append(w)
    return out

generated = generate_bigram("the", 20)
print("Generated (20 tokens, start=the):")
print(" ", " ".join(generated))


**N-gram models are simple but limited** — they have **no long-range memory**: only the previous $(n-1)$ tokens (here, **one** word) shape the next-word distribution.


## Section 4: Perplexity — How Confused Is the Model?

**Perplexity** $= 2^{H}$ when $H$ is cross-entropy in bits. With natural logs: $\mathrm{PP} = \exp(-\frac{1}{N}\sum_i \log P(w_i|\ldots))$.

A perplexity of **50** means the model is about as uncertain as choosing uniformly among **50** words at each step.


In [ ]:
# Add-k smoothing for bigrams (avoids log(0))
k_sm = 0.01

def P_bigram_smooth(w2, w1):
    denom = unigram_context[w1] + k_sm * V
    num = bigram_counts[(w1, w2)] + k_sm
    return num / denom

def perplexity_bigram_sentence(sentence, verbose=True):
    toks = tokenize(sentence)
    n = len(toks) - 1
    if n <= 0:
        return float('nan')
    log_sum = 0.0
    for i in range(n):
        w1, w2 = toks[i], toks[i + 1]
        p = P_bigram_smooth(w2, w1)
        log_p = math.log(p)
        log_sum += log_p
        if verbose:
            print(f'  P("{w2}" | "{w1}") = {p:.6f}   log2 p = {log_p/math.log(2):.4f} bits')
    avg_log = log_sum / n
    avg_log2 = avg_log / math.log(2)
    pp = math.exp(-avg_log)
    if verbose:
        print(f"Average neg log-prob (nats/token): {-avg_log:.6f}")
        print(f"Cross-entropy (bits/token): {avg_log2:.4f}")
        print(f"Perplexity = exp(-avg log p) = {pp:.4f}")
    return pp

test_sentence = "the cat sat on the mat"
print("Test sentence:", repr(test_sentence))
print("Bigram perplexity (step-by-step):")
pp_bi = perplexity_bigram_sentence(test_sentence, verbose=True)


In [ ]:
def perplexity_unigram_sentence(sentence):
    toks = tokenize(sentence)
    log_sum = sum(math.log(P_unigram(w)) for w in toks)
    n = len(toks)
    return math.exp(-log_sum / n)

pp_uni = perplexity_unigram_sentence(test_sentence)
print("Same test sentence:", repr(test_sentence))
print(f"Unigram perplexity: {pp_uni:.4f}")
print(f"Bigram perplexity:   {pp_bi:.4f}")


## Section 5: One-Hot Vectors — The Problem

**One-hot encoding:** each word gets one coordinate. **Distinct words are orthogonal** — cosine similarity is **0** unless they are identical.


In [ ]:
words_5 = ["cat", "dog", "mat", "the", "sat"]
d = len(words_5)

def one_hot(i, dim=d):
    v = np.zeros(dim)
    v[i] = 1.0
    return v

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

print("Cosine similarity between one-hot vectors (different words -> 0):")
for i in range(d):
    for j in range(i + 1, d):
        wi, wj = words_5[i], words_5[j]
        s = cosine_sim(one_hot(i), one_hot(j))
        print(f"  sim({wi}, {wj}) = {s:.4f}")


We want **dense embeddings** so that **related words are nearby** in cosine similarity.


## Section 6: Word2Vec Skip-Gram from Scratch

**Skip-gram:** predict **context** words from a **target** word within a window. **Negative sampling:** classify real (target, context) pairs vs random noise pairs using **binary cross-entropy** on dot-product logits.


In [ ]:
sg_corpus = [
    "cats and dogs are friendly pets",
    "dogs love to play fetch in the park",
    "cats enjoy napping in sunny spots",
    "fresh fish is a treat for cats",
    "dogs often eat kibble or meat",
    "the kitchen smells like warm food",
    "puppies and kittens are adorable animals",
    "birds sing while dogs bark outside",
    "a bowl of milk may attract a cat",
    "training dogs takes patience and treats",
    "shelter pets need love and care",
    "organic food is popular for pets",
]

def sg_tokenize(text):
    return text.lower().split()

sg_word_counts = Counter()
for line in sg_corpus:
    sg_word_counts.update(sg_tokenize(line))

sg_vocab = sorted(sg_word_counts.keys())
word2idx = {w: i for i, w in enumerate(sg_vocab)}
idx2word = {i: w for w, i in word2idx.items()}
Vs = len(sg_vocab)
print("Vocabulary size:", Vs)

window_size = 2
pairs = []
for line in sg_corpus:
    toks = sg_tokenize(line)
    L = len(toks)
    for i in range(L):
        ti = word2idx[toks[i]]
        for j in range(max(0, i - window_size), min(L, i + window_size + 1)):
            if j == i:
                continue
            pairs.append((ti, word2idx[toks[j]]))

print("Number of (target, context) pairs:", len(pairs))
print("Example index pairs:", pairs[:6])


In [ ]:
class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.emb_t = nn.Embedding(vocab_size, emb_dim)
        self.emb_c = nn.Embedding(vocab_size, emb_dim)
        nn.init.uniform_(self.emb_t.weight, -0.5 / emb_dim, 0.5 / emb_dim)
        nn.init.uniform_(self.emb_c.weight, -0.5 / emb_dim, 0.5 / emb_dim)

    def forward(self, target, pos_context, neg_context):
        t = self.emb_t(target)
        c_pos = self.emb_c(pos_context)
        pos_score = (t * c_pos).sum(dim=1)
        loss_pos = F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
        c_neg = self.emb_c(neg_context)
        neg_score = (t.unsqueeze(1) * c_neg).sum(dim=2)
        loss_neg = F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
        return loss_pos + loss_neg.mean()

emb_dim = 16
num_neg = 8
model_sg = SkipGramNegSampling(Vs, emb_dim)
opt = optim.Adam(model_sg.parameters(), lr=0.02)

def batch_iter(pairs_list, batch_size=64):
    idxs = list(range(len(pairs_list)))
    random.shuffle(idxs)
    for s in range(0, len(idxs), batch_size):
        batch = [pairs_list[i] for i in idxs[s : s + batch_size]]
        if not batch:
            continue
        t = torch.tensor([a for a, _ in batch], dtype=torch.long)
        c = torch.tensor([b for _, b in batch], dtype=torch.long)
        neg = torch.randint(0, Vs, (len(batch), num_neg))
        yield t, c, neg

n_epochs = 100
for epoch in range(1, n_epochs + 1):
    total_loss = 0.0
    n_batches = 0
    for t, c, neg in batch_iter(pairs):
        opt.zero_grad()
        loss = model_sg(t, c, neg)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        n_batches += 1
    if epoch == 1 or epoch % 20 == 0 or epoch == n_epochs:
        print(f"Epoch {epoch:3d}  avg loss = {total_loss / max(n_batches, 1):.4f}")


In [ ]:
def tensor_cosine(a, b):
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())

emb_weight = model_sg.emb_t.weight.detach()

def sim_words(w1, w2):
    i1, i2 = word2idx[w1], word2idx[w2]
    return tensor_cosine(emb_weight[i1], emb_weight[i2])

pairs_show = [("cats", "dogs"), ("cats", "food"), ("dogs", "park"), ("fish", "cats")]
print("Cosine similarity (learned target embeddings):")
for a, b in pairs_show:
    if a in word2idx and b in word2idx:
        print(f"  sim({a}, {b}) = {sim_words(a, b):.4f}")

if PCA is None:
    print("Install scikit-learn for PCA: pip install scikit-learn")
else:
    X = emb_weight.cpu().numpy()
    pca = PCA(n_components=2)
    Z = pca.fit_transform(X)
    plt.figure(figsize=(9, 7))
    plt.scatter(Z[:, 0], Z[:, 1], alpha=0.6, label="words")
    for i, w in idx2word.items():
        plt.annotate(w, (Z[i, 0], Z[i, 1]), fontsize=8)
    plt.title("Skip-gram target embeddings (2D PCA)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Section 7: Pretrained Embeddings with Gensim

Large corpora yield stronger semantics. Load **`glove-wiki-gigaword-50`** via `gensim.downloader` (downloads on first run).


In [ ]:
import gensim.downloader as api

print("Loading glove-wiki-gigaword-50 (first run may download ~66MB)...")
pretrained = api.load("glove-wiki-gigaword-50")

for seed in ["king", "computer", "happy"]:
    print(f"\nMost similar to {seed!r}:")
    for w, score in pretrained.most_similar(seed, topn=8):
        print(f"  {w:15s}  {score:.4f}")

print("\nEmbedding arithmetic: king - man + woman  (top 5)")
print(pretrained.most_similar(positive=["king", "woman"], negative=["man"], topn=5))

odd = pretrained.doesnt_match(["breakfast", "lunch", "dinner", "computer"])
print("\nOdd one out:", odd)


## Section 8: BPE Tokenization with Hugging Face `transformers`

Modern LLMs use **subword** tokenization (e.g. **BPE**). Unknown or rare words split into pieces, reducing out-of-vocabulary failures.


In [ ]:
from transformers import GPT2TokenizerFast

tok_hf = GPT2TokenizerFast.from_pretrained("gpt2")

examples = [
    "The quick brown fox jumps.",
    "uncharacteristically",
    "antidisestablishmentarianism",
    "madeupwordxyz123",
    "low-frequency-term-🙂",
]

print("Vocabulary size:", tok_hf.vocab_size)
for ex in examples:
    ids = tok_hf.encode(ex, add_special_tokens=False)
    pieces = tok_hf.convert_ids_to_tokens(ids)
    print("\nTEXT:", repr(ex))
    print("  tokens:", pieces)
    print("  ids   :", ids)

roundtrip = tok_hf.decode(tok_hf.encode(examples[-1], add_special_tokens=False))
print("\nDecode round-trip last example:", repr(roundtrip))


## Section 9: Perplexity with GPT-2

A **real** language model assigns token probabilities. **Cross-entropy loss** on next-token prediction yields **perplexity** $\approx \exp(\text{loss})$.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
gpt_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2.eval()

def gpt2_perplexity(text):
    inputs = gpt_tok(text, return_tensors="pt")
    with torch.no_grad():
        out = gpt2(**inputs, labels=inputs["input_ids"])
    loss = out.loss
    ppl = math.exp(loss.item())
    return ppl, loss.item()

s_good = "The cat sat on the mat and looked outside."
s_bad = "Quantum breakfast computes the purple seven."

ppl_good, loss_good = gpt2_perplexity(s_good)
ppl_bad, loss_bad = gpt2_perplexity(s_bad)

print("Well-formed:", repr(s_good))
print(f"  loss={loss_good:.4f}  perplexity={ppl_good:.2f}")
print("Nonsensical:", repr(s_bad))
print(f"  loss={loss_bad:.4f}  perplexity={ppl_bad:.2f}")
print("\nLower perplexity usually indicates more plausible text under the model.")


This is **exactly how LLMs are evaluated** on language modeling benchmarks: **lower perplexity** means a better language model (under the same tokenizer and test data).


## Section 10: Key Takeaways

- **Chain rule** factorizes sequence probability; **n-grams** estimate $P(w_t|w_{t-1})$ from counts (use **smoothing** in practice).
- **Perplexity** aggregates uncertainty: $\mathrm{PP} = \exp(-\frac{1}{N}\sum \log P)$.
- **One-hot** vectors cannot express similarity; **embeddings** learn similarity from data.
- **Skip-gram + negative sampling** scales learning of word vectors.
- **Pretrained** GloVe/Word2Vec capture broad semantics; **BPE** handles rare words as subwords.
- **GPT-2** shows much lower perplexity on plausible English than on scrambled text.
